# Workshop : introduction au Machine Learning

Ce notebook reprend les notions de l'application du cours, **avec le code visible**.
Exécutez les cellules dans l'ordre avec `Shift + Entrée`, modifiez, relancez : c'est fait pour jouer.

1. Mise en place
2. Le café : une régression linéaire à la main
3. Sous-apprentissage et sur-apprentissage
4. Préparation des données OMNI
5. Évaluation d'un modèle
6. Un réseau de neurones (MLP) en PyTorch
7. À vous de jouer

## 1. Mise en place

Le code commun à l'application et au notebook est dans le dossier `utils/` du projet :
`data.py` (données), `models.py` (modèles) et `plots.py` (graphiques).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score, recall_score)
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Le notebook est dans notebook/ : on ajoute le dossier du projet pour pouvoir importer utils/
ROOT = Path.cwd() if (Path.cwd() / "utils").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from utils import data as D
from utils import models as M
from utils import plots as P

# Seeds fixées : mêmes résultats à chaque exécution
SEED = 42
M.set_seed(SEED)

print("PyTorch", torch.__version__)

## 2. Le café : une régression linéaire à la main

On veut prédire les ventes de café à partir de la température, avec une droite **ventes = a × température + b**.
Le jour 7, la vente n'a pas été notée : c'est ce qu'on veut prédire.

In [ ]:
coffee = D.coffee_data()
x, y = D.coffee_xy()  # températures et ventes des jours connus (sans le jour 7)
coffee

### La descente de gradient, écrite à la main

On part de a = 0 et b = 0, puis on répète : prédire, mesurer l'erreur, calculer la pente de l'erreur
(le gradient), faire un petit pas dans la direction qui descend.
Comme dans l'app (module 1.5), on travaille sur la température **normalisée**.

In [ ]:
def coffee_gradient_descent(learning_rate=0.1, n_iterations=100, normalize=True):
    """Descente de gradient sur la droite y = a*x + b. Renvoie a, b (en °C) et l'erreur à chaque itération."""
    # Normalisation : x' = (x - moyenne) / écart-type
    x_mean, x_std = (x.mean(), x.std()) if normalize else (0.0, 1.0)
    x_work = (x - x_mean) / x_std

    a, b = 0.0, 0.0  # poids de départ
    losses = []
    for iteration in range(n_iterations):
        y_pred = a * x_work + b                # 1. prédiction avec les poids actuels
        error = y_pred - y                     # 2. erreur sur chaque jour
        loss = np.mean(error**2)               # 3. erreur quadratique moyenne (MSE)
        grad_a = 2 * np.mean(error * x_work)   # 4. gradient : pente de la MSE selon a...
        grad_b = 2 * np.mean(error)            #    ... et selon b
        a = a - learning_rate * grad_a         # 5. un pas dans la direction qui descend
        b = b - learning_rate * grad_b
        losses.append(loss)

    # Retour à la température en °C : y = a'*(x - moyenne)/écart-type + b'
    return a / x_std, b - a * x_mean / x_std, losses


a, b, losses = coffee_gradient_descent(learning_rate=0.1, n_iterations=100)
print(f"Descente de gradient : a = {a:.3f}, b = {b:.2f}, MSE finale = {losses[-1]:.1f}")
P.loss_curve(losses, title="Erreur au fil des itérations").show()

### Comparaison avec scikit-learn

`LinearRegression` trouve directement la meilleure droite (moindres carrés). On doit retrouver les mêmes valeurs.

In [ ]:
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)  # scikit-learn attend un tableau 2D : une ligne par exemple, une colonne par feature
print(f"scikit-learn          : a = {model.coef_[0]:.3f}, b = {model.intercept_:.2f}")

temperature_day7 = 22
print(f"Prédiction du jour 7 ({temperature_day7} °C) : {model.predict([[temperature_day7]])[0]:.0f} tasses")
P.coffee_plot(coffee, line=(model.coef_[0], model.intercept_), residuals=True).show()

## 3. Sous-apprentissage et sur-apprentissage

On ajuste des polynômes de degré croissant sur des points bruités (train),
et on mesure l'erreur sur d'autres points (validation).

In [ ]:
curve = D.noisy_curve_data()
errors = M.poly_errors(curve, degrees=range(1, 16))
best_degree = int(errors.loc[errors["validation"].idxmin(), "degre"])
print(f"Meilleur degré (erreur de validation la plus faible) : {best_degree}")
errors.round(3)

In [ ]:
P.poly_error_plot(errors, best_degree).show()
for degree in [1, best_degree, 15]:
    model = M.fit_poly(curve["x_train"], curve["y_train"], degree)
    P.poly_fit_plot(curve, model, degree, curve=D.true_curve).show()

L'erreur de **train** baisse toujours avec la complexité. L'erreur de **validation** baisse, puis remonte :
le modèle apprend le bruit. On choisit donc le modèle sur l'erreur de validation.

## 4. Préparation des données OMNI

Données horaires de la NASA (2015 à 2024) : vent solaire (V, Bz, N) et indices (Kp, Dst, F10.7).
Tâche du fil rouge : **prédire Kp de l'heure suivante** à partir du vent solaire et de l'heure.

### 4.1 Données brutes et valeurs de remplissage

In [ ]:
raw = D.load_omni_raw()
print(raw.shape)
raw.head()

In [ ]:
# OMNI écrit une valeur de remplissage (ex. 9999 pour V) quand il n'y a pas de mesure
for column, fill_value in D.FILL_VALUES.items():
    n_fill = (raw[column] >= fill_value).sum()
    print(f"{column:5s} : {n_fill:4d} valeurs de remplissage ({fill_value})")

### 4.2 Nettoyage : valeurs de remplissage → NaN, et petits trous interpolés

In [ ]:
omni = D.clean_omni(raw)          # valeurs de remplissage -> NaN, Kp divisé par 10
print("NaN après nettoyage :", omni[["v", "bz", "n"]].isna().sum().to_dict())

omni = D.fill_small_gaps(omni)    # interpolation des trous de 3 h ou moins
print("NaN restants        :", omni[["v", "bz", "n"]].isna().sum().to_dict())

### 4.3 Encodage cyclique de l'heure et du jour de l'année (cos, sin)

In [ ]:
omni = D.add_cyclic_features(omni)
omni[["time", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]].head(4)

### 4.4 La cible : Kp de l'heure suivante

`shift(-1)` décale la colonne d'une ligne vers le haut : sur la ligne de 10h, on trouve le Kp de 11h.
On le fait **avant** de retirer les lignes incomplètes, pour que les heures se suivent bien.

In [ ]:
omni = D.add_targets(omni)  # colonnes kp_next (régression) et storm_next (orage : Kp >= 5)
omni[["time", "kp", "kp_next", "storm_next"]].head(4)

In [ ]:
# On retire les heures où il manque une feature ou la cible (grands trous)
dataset = omni.dropna(subset=D.FEATURES + [D.TARGET]).reset_index(drop=True)
print(f"{len(dataset)} heures utilisables sur {len(omni)}")
print("Features :", D.FEATURES)

# Vérification : on obtient bien les mêmes données que l'app
pd.testing.assert_frame_equal(dataset, D.load_dataset())

### 4.5 Split temporel

Train 2015 à 2021, validation 2022, test 2023 à 2024 (le test contient l'orage de mai 2024).
Surtout pas de split aléatoire avec des séries temporelles (module 4.1 de l'app).

In [ ]:
train, val, test = D.temporal_split(dataset)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s} : {len(part):6d} heures, {part[D.STORM_TARGET].mean():.1%} d'heures d'orage")

### 4.6 Standardisation, avec les statistiques du **train uniquement**

In [ ]:
mean, std = D.fit_scaler(train, D.FEATURES)   # moyenne et écart-type calculés sur le train
X_train = D.apply_scaler(train, mean, std)     # puis appliqués tels quels aux trois jeux
X_val = D.apply_scaler(val, mean, std)
X_test = D.apply_scaler(test, mean, std)

pd.DataFrame({"moyenne train": X_train.mean(), "moyenne test": X_test.mean()}).round(2)

La moyenne du train vaut 0 ; celle du test non, et c'est normal : le test n'a pas servi à calculer la normalisation.

## 5. Évaluation d'un modèle

On règle et compare les modèles sur la **validation**. Le test est gardé pour la toute fin (section 6.5).

### 5.1 Régression : MAE, RMSE, R²

In [ ]:
def regression_scores(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),          # erreur moyenne, en points de Kp
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),  # pénalise davantage les grosses erreurs
        "R²": r2_score(y_true, y_pred),                       # 1 = parfait, 0 = pas mieux que la moyenne
    }


y_train, y_val, y_test = train[D.TARGET], val[D.TARGET], test[D.TARGET]

forest = M.train_forest(X_train, y_train)  # random forest scikit-learn (60 arbres)
scores = {"random forest": regression_scores(y_val, forest.predict(X_val))}
pd.DataFrame(scores).T.round(3)

### 5.2 Classification « orage ou pas » : accuracy, précision, rappel

In [ ]:
s_train, s_val, s_test = train[D.STORM_TARGET], val[D.STORM_TARGET], test[D.STORM_TARGET]

logreg = M.train_logreg(X_train, s_train)
proba_val = logreg.predict_proba(X_val)[:, 1]  # probabilité d'orage
threshold = 0.5
pred_val = (proba_val >= threshold).astype(int)

print(f"Accuracy  : {accuracy_score(s_val, pred_val):.1%}")
print(f"Précision : {precision_score(s_val, pred_val):.1%}  (quand j'annonce un orage, ai-je raison ?)")
print(f"Rappel    : {recall_score(s_val, pred_val):.1%}  (parmi les vrais orages, combien en ai-je vu ?)")

tn, fp, fn, tp = confusion_matrix(s_val, pred_val).ravel()
P.confusion_matrix_plot(tp, fp, fn, tn).show()

In [ ]:
# Le piège de l'accuracy : un « modèle » qui annonce toujours calme
always_calm = np.zeros_like(pred_val)
print(f"Toujours calme : accuracy {accuracy_score(s_val, always_calm):.1%}, rappel {recall_score(s_val, always_calm):.0%}")

## 6. Un réseau de neurones (MLP) en PyTorch

Le cœur du notebook. Quatre briques :
1. un `Dataset` qui renvoie un exemple (features, cible) ;
2. un `DataLoader` qui regroupe les exemples en paquets (**batches**) ;
3. le modèle, une classe `nn.Module` ;
4. la boucle d'entraînement.

### 6.1 Dataset et DataLoader

In [ ]:
class OmniDataset(Dataset):
    """Un exemple = (features d'une heure, cible)."""

    def __init__(self, features: pd.DataFrame, target: pd.Series):
        # PyTorch travaille avec des tenseurs float32
        self.X = torch.tensor(features.to_numpy(), dtype=torch.float32)
        self.y = torch.tensor(target.to_numpy(), dtype=torch.float32).unsqueeze(1)  # forme (n, 1)

    def __len__(self):
        return len(self.X)  # nombre d'exemples

    def __getitem__(self, i):
        return self.X[i], self.y[i]  # l'exemple numéro i


def make_loaders(target_train, target_val, batch_size=256):
    generator = torch.Generator().manual_seed(SEED)  # mélange reproductible
    train_loader = DataLoader(OmniDataset(X_train, target_train), batch_size=batch_size, shuffle=True,
                              generator=generator)  # on mélange le train à chaque epoch
    val_loader = DataLoader(OmniDataset(X_val, target_val), batch_size=4096)  # pas besoin de mélanger
    return train_loader, val_loader


train_loader, val_loader = make_loaders(y_train, y_val)
X_batch, y_batch = next(iter(train_loader))
print("Un batch :", X_batch.shape, "->", y_batch.shape)

### 6.2 Le modèle

In [ ]:
class KpMLP(nn.Module):
    """MLP : 7 features -> couche cachée -> couche cachée -> 1 sortie."""

    def __init__(self, n_inputs: int, n_hidden: int = 64):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_inputs, n_hidden),  # somme pondérée + biais, pour n_hidden neurones
            nn.ReLU(),                      # activation
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1),         # une seule sortie : Kp prédit
        )

    def forward(self, x):
        return self.layers(x)


M.set_seed(SEED)
model = KpMLP(n_inputs=len(D.FEATURES))
print(model)
print("Nombre de poids :", sum(p.numel() for p in model.parameters()))

### 6.3 La boucle d'entraînement, ligne par ligne

Pour chaque **epoch** (un passage sur tout le train), et pour chaque **batch** :
1. **forward** : le modèle prédit ;
2. **loss** : on mesure l'erreur ;
3. **zero_grad** : on efface les gradients du pas précédent (sinon PyTorch les additionne) ;
4. **backward** : PyTorch calcule le gradient de la loss pour chaque poids ;
5. **step** : l'optimiseur fait un pas de descente de gradient.

In [ ]:
def evaluate_loss(model, loss_fn, loader):
    """Loss moyenne sur un jeu de données, sans entraîner."""
    model.eval()                   # mode évaluation
    total = 0.0
    with torch.no_grad():          # pas besoin de gradients pour évaluer : plus rapide
        for X_batch, y_batch in loader:
            total += loss_fn(model(X_batch), y_batch).item() * len(X_batch)
    return total / len(loader.dataset)


def train_model(model, loss_fn, train_loader, val_loader, n_epochs=15, learning_rate=1e-3):
    # Adam : une variante de la descente de gradient qui adapte la taille du pas pour chaque poids
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train": [], "val": []}

    for epoch in range(n_epochs):                  # une epoch = un passage complet sur le train
        model.train()                              # mode entraînement
        total = 0.0
        for X_batch, y_batch in train_loader:      # le train, paquet par paquet
            y_pred = model(X_batch)                # 1. forward : le modèle prédit
            loss = loss_fn(y_pred, y_batch)        # 2. loss : l'erreur sur ce batch
            optimizer.zero_grad()                  # 3. on efface les gradients du pas précédent
            loss.backward()                        # 4. backward : gradient de la loss pour chaque poids
            optimizer.step()                       # 5. step : un pas de descente de gradient
            total += loss.item() * len(X_batch)    # pour la loss moyenne de l'epoch

        history["train"].append(total / len(train_loader.dataset))
        history["val"].append(evaluate_loss(model, loss_fn, val_loader))
        print(f"epoch {epoch + 1:2d} : loss train {history['train'][-1]:.3f}, validation {history['val'][-1]:.3f}")

    return history


def predict(model, features: pd.DataFrame) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(features.to_numpy(), dtype=torch.float32)).numpy().ravel()

### 6.4 Régression de Kp

In [ ]:
M.set_seed(SEED)
model = KpMLP(n_inputs=len(D.FEATURES))
history = train_model(model, nn.MSELoss(), train_loader, val_loader, n_epochs=15)

fig = P.loss_curve(history["train"], val_losses=history["val"], name="train", title="Loss au fil des epochs")
fig.update_xaxes(title="epoch")
fig.show()

In [ ]:
scores["MLP"] = regression_scores(y_val, predict(model, X_val))
pd.DataFrame(scores).T.round(3)  # comparaison sur la validation

### 6.5 Variante classification : orage ou pas

Même réseau, mais la sortie est un **logit** (un score) : la fonction sigmoïde le transforme en probabilité.
La loss adaptée est l'entropie croisée binaire, `BCEWithLogitsLoss`, qui applique la sigmoïde elle-même.

In [ ]:
storm_train_loader, storm_val_loader = make_loaders(s_train, s_val)

M.set_seed(SEED)
storm_model = KpMLP(n_inputs=len(D.FEATURES))
storm_history = train_model(storm_model, nn.BCEWithLogitsLoss(), storm_train_loader, storm_val_loader, n_epochs=10)


def storm_probability(model, features):
    return torch.sigmoid(torch.tensor(predict(model, features))).numpy()  # logit -> probabilité

**Choix du seuil, sur la validation.** On regarde précision et rappel pour plusieurs seuils,
et on retient celui qui maximise le **F1** (une moyenne qui équilibre précision et rappel).

In [ ]:
proba_val = storm_probability(storm_model, X_val)
rows = []
for t in np.arange(0.05, 0.96, 0.05):
    pred = (proba_val >= t).astype(int)
    rows.append({"seuil": round(t, 2),
                 "précision": precision_score(s_val, pred, zero_division=0),
                 "rappel": recall_score(s_val, pred),
                 "F1": f1_score(s_val, pred)})
thresholds = pd.DataFrame(rows)
best_threshold = float(thresholds.loc[thresholds["F1"].idxmax(), "seuil"])
print(f"Seuil retenu : {best_threshold}")
thresholds.round(2)

### 6.6 Évaluation finale sur le test, **une seule fois**

Tous les choix sont faits (architecture, epochs, seuil). On ouvre enfin le test.

In [ ]:
print("Régression de Kp (test 2023 à 2024) :")
for name, value in regression_scores(y_test, predict(model, X_test)).items():
    print(f"  {name:4s} = {value:.3f}")

proba_test = storm_probability(storm_model, X_test)
pred_test = (proba_test >= best_threshold).astype(int)
print(f"\nOrage ou pas (seuil {best_threshold}) :")
print(f"  accuracy {accuracy_score(s_test, pred_test):.1%}, précision {precision_score(s_test, pred_test):.1%}, "
      f"rappel {recall_score(s_test, pred_test):.1%}")

tn, fp, fn, tp = confusion_matrix(s_test, pred_test).ravel()
P.confusion_matrix_plot(tp, fp, fn, tn).show()

may = test["time"].between("2024-05-08", "2024-05-15").to_numpy()
P.storm_probability_plot(test[may], proba_test[may], best_threshold).show()

## 7. À vous de jouer

Pour chaque exercice : lisez la consigne, écrivez votre code dans la cellule en dessous, et comparez avec
les résultats obtenus plus haut. Les pistes en commentaire vous aident à démarrer.

### Exercice 1 : le learning rate

Relancez la descente de gradient du café (section 2) avec un learning rate trop grand (1.1), puis trop petit (0.001).
Que devient l'erreur ? Et sans normalisation (`normalize=False`), à partir de quel learning rate ça diverge ?

In [ ]:
# Piste : a, b, losses = coffee_gradient_descent(learning_rate=..., n_iterations=100, normalize=...)
#         P.loss_curve(losses).show()

### Exercice 2 : retirer l'encodage cos / sin

Entraînez la random forest (section 5.1) avec seulement `["v", "bz", "n"]` comme features.
Le score de validation change-t-il ? Essayez aussi l'heure brute (`train["time"].dt.hour`) comme feature.

In [ ]:
# Piste : features = ["v", "bz", "n"]
#         forest_small = M.train_forest(X_train[features], y_train)
#         regression_scores(y_val, forest_small.predict(X_val[features]))

### Exercice 3 : split aléatoire

Découpez avec `D.split_labels_random(dataset)` au lieu du split temporel, entraînez la random forest,
et comparez le R² sur le « test ». Pourquoi est-il meilleur ? Peut-on s'y fier ?

In [ ]:
# Piste : labels = D.split_labels_random(dataset)
#         train_r, test_r = dataset[labels == "train"], dataset[labels == "test"]
#         forest_r = M.train_forest(train_r[D.FEATURES], train_r[D.TARGET])
#         regression_scores(test_r[D.TARGET], forest_r.predict(test_r[D.FEATURES]))

### Exercice 4 : normaliser sur tout le jeu

Calculez la moyenne et l'écart-type sur **tout** le jeu (train + validation + test) au lieu du train seul.
Comparez les statistiques avec celles du train. Pourquoi est-ce une fuite d'information, même si l'effet semble petit ?

In [ ]:
# Piste : mean_all, std_all = D.fit_scaler(dataset, D.FEATURES)
#         pd.DataFrame({"train": mean, "tout le jeu": mean_all})

### Exercice 5 : la taille du réseau

Changez `n_hidden` (par exemple 4, puis 256) ou le nombre d'epochs du MLP (section 6.4).
Observez les courbes de loss train et validation : voyez-vous du sous-apprentissage ou du sur-apprentissage ?

In [ ]:
# Piste : M.set_seed(SEED)
#         model_test = KpMLP(n_inputs=len(D.FEATURES), n_hidden=...)
#         history_test = train_model(model_test, nn.MSELoss(), train_loader, val_loader, n_epochs=...)

### Exercice 6 : le seuil de classification

Sur la **validation**, faites varier le seuil du classifieur d'orages (section 6.5) et affichez
précision, rappel et la matrice de confusion. Quel seuil choisiriez-vous si manquer un orage coûtait très cher ?

In [ ]:
# Piste : pred = (proba_val >= 0.2).astype(int)
#         precision_score(s_val, pred), recall_score(s_val, pred)